
# **NYC Taxi Trip Analytics using Apache Spark**

## **Part 1 – Environment Setup**

In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = (
    SparkSession.builder \
    .appName("NYC Taxi Assignment") \
    .master("local[*]") \
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/02 21:06:59 WARN Utils: Your hostname, asrar-Latitude-3510, resolves to a loopback address: 127.0.1.1; using 192.168.43.254 instead (on interface wlp0s20f3)
26/08/02 21:06:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/02 21:07:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
print("Spark Version  :", spark.version)
print("Spark Master   :", spark.sparkContext.master)
print("Application    :", spark.sparkContext.appName)

Spark Version  : 4.2.0
Spark Master   : local[*]
Application    : NYC Taxi Assignment


## **Part 2 – Load Dataset**

In [6]:
df = spark.read.parquet("yellow_tripdata_2024-01.parquet")

In [7]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [8]:
num_columns = len(df.columns)
num_records = df.count()

print(f"Number of Columns : {num_columns}")
print(f"Number of Records : {num_records}")

Number of Columns : 19
Number of Records : 2964624


In [9]:
df.show(20, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

## **Part 3 – Exploratory Data Analysis (EDA)**

In [10]:
from pyspark.sql.functions import (
    avg,
    min,
    max,
    countDistinct,
    round
)

In [11]:
eda_summary = df.select(
    countDistinct("VendorID").alias("Unique Vendors"),
    round(avg("trip_distance"), 2).alias("Average Trip Distance"),
    round(avg("fare_amount"), 2).alias("Average Fare"),
    max("fare_amount").alias("Maximum Fare"),
    min("fare_amount").alias("Minimum Fare"),
    round(avg("passenger_count"), 2).alias("Average Passenger Count"),
    countDistinct("payment_type").alias("Payment Methods")
)

In [12]:
eda_summary.show(truncate=False)

+--------------+---------------------+------------+------------+------------+-----------------------+---------------+
|Unique Vendors|Average Trip Distance|Average Fare|Maximum Fare|Minimum Fare|Average Passenger Count|Payment Methods|
+--------------+---------------------+------------+------------+------------+-----------------------+---------------+
|3             |3.65                 |18.18       |5000.0      |-899.0      |1.34                   |5              |
+--------------+---------------------+------------+------------+------------+-----------------------+---------------+



In [13]:
total_trips = df.count()

trip_dates = df.select(
    min("tpep_pickup_datetime").alias("Earliest Trip"),
    max("tpep_pickup_datetime").alias("Latest Trip")
).first()

In [14]:
print(f"Total Trips      : {total_trips}")
print(f"Earliest Trip    : {trip_dates['Earliest Trip']}")
print(f"Latest Trip      : {trip_dates['Latest Trip']}")

Total Trips      : 2964624
Earliest Trip    : 2002-12-31 22:59:39
Latest Trip      : 2024-02-01 00:01:15


## **Part 4 – Data Cleaning**

In [15]:
from pyspark.sql.functions import col

In [16]:
clean_df = (
    df
    .dropDuplicates()
    .filter(col("trip_distance") > 0)
    .filter(col("fare_amount") >= 0)
    .na.drop()
)

In [17]:
print("Cleaned Dataset")

print("Records After Cleaning:", clean_df.count())

Cleaned Dataset


Records After Cleaning: 2754895


In [18]:
clean_df.show(20, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:47:21 |2024-01-01 00:51:58  |2              |1.13         |1         |N                 |238         |166         |1           |7.9        |1.0  |0.5    |2.58     

## **Part 5 – Spark Transformations**

### **Filter**

In [19]:
filtered_df = clean_df.filter(
    col("trip_distance") > 2
)

filtered_df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:42:51 |2024-01-01 01:16:30  |1              |4.43         |1         |N                 |144         |142         |1           |31.0       |1.0  |0.5    |7.2      

### **Select**

In [20]:
selected_df = filtered_df.select(
    "VendorID",
    "trip_distance",
    "fare_amount",
    "payment_type",
    "PULocationID"
)

selected_df.show(5, truncate=False)

+--------+-------------+-----------+------------+------------+
|VendorID|trip_distance|fare_amount|payment_type|PULocationID|
+--------+-------------+-----------+------------+------------+
|2       |4.43         |31.0       |1           |144         |
|2       |3.64         |17.7       |2           |41          |
|2       |5.9          |30.3       |1           |114         |
|2       |3.43         |24.7       |1           |90          |
|2       |2.25         |14.2       |1           |144         |
+--------+-------------+-----------+------------+------------+
only showing top 5 rows


### **withColumn**

In [21]:
with_column_df = selected_df.withColumn(
    "Fare_With_Tax",
    round(col("fare_amount") * 1.10, 2)
)

with_column_df.show(5, truncate=False)

+--------+-------------+-----------+------------+------------+-------------+
|VendorID|trip_distance|fare_amount|payment_type|PULocationID|Fare_With_Tax|
+--------+-------------+-----------+------------+------------+-------------+
|2       |4.43         |31.0       |1           |144         |34.1         |
|2       |3.64         |17.7       |2           |41          |19.47        |
|2       |5.9          |30.3       |1           |114         |33.33        |
|2       |3.43         |24.7       |1           |90          |27.17        |
|2       |2.25         |14.2       |1           |144         |15.62        |
+--------+-------------+-----------+------------+------------+-------------+
only showing top 5 rows


### **orderBy**

In [22]:
ordered_df = with_column_df.orderBy(
    col("Fare_With_Tax").desc()
)

ordered_df.show(5, truncate=False)

+--------+-------------+-----------+------------+------------+-------------+
|VendorID|trip_distance|fare_amount|payment_type|PULocationID|Fare_With_Tax|
+--------+-------------+-----------+------------+------------+-------------+
|2       |31.95        |2221.3     |2           |220         |2443.43      |
|2       |233.25       |1616.5     |2           |168         |1778.15      |
|2       |142.62       |912.3      |2           |132         |1003.53      |
|2       |157.25       |899.0      |2           |265         |988.9        |
|2       |109.75       |761.1      |2           |161         |837.21       |
+--------+-------------+-----------+------------+------------+-------------+
only showing top 5 rows


### **Drop**

In [23]:
dropped_df = ordered_df.drop(
    "payment_type"
)

dropped_df.show(5, truncate=False)

+--------+-------------+-----------+------------+-------------+
|VendorID|trip_distance|fare_amount|PULocationID|Fare_With_Tax|
+--------+-------------+-----------+------------+-------------+
|2       |31.95        |2221.3     |220         |2443.43      |
|2       |233.25       |1616.5     |168         |1778.15      |
|2       |142.62       |912.3      |132         |1003.53      |
|2       |157.25       |899.0      |265         |988.9        |
|2       |109.75       |761.1      |161         |837.21       |
+--------+-------------+-----------+------------+-------------+
only showing top 5 rows


### **Distinct**

In [24]:
distinct_df = dropped_df.distinct()

distinct_df.show(5, truncate=False)

+--------+-------------+-----------+------------+-------------+
|VendorID|trip_distance|fare_amount|PULocationID|Fare_With_Tax|
+--------+-------------+-----------+------------+-------------+
|1       |5.4          |28.2       |211         |31.02        |
|2       |10.45        |42.9       |138         |47.19        |
|2       |3.16         |14.9       |132         |16.39        |
|2       |13.42        |54.8       |195         |60.28        |
|2       |3.62         |24.0       |140         |26.4         |
+--------+-------------+-----------+------------+-------------+
only showing top 5 rows


### **groupBy**

In [25]:
grouped_df = distinct_df.groupBy(
    "VendorID"
).agg(
    round(
        avg("Fare_With_Tax"),
        2
    ).alias("Average Fare")
)

grouped_df.show()

+--------+------------+
|VendorID|Average Fare|
+--------+------------+
|       1|       40.12|
|       2|       38.27|
+--------+------------+



### **Alias**

In [26]:
aliased_df = grouped_df.select(
    col("VendorID").alias("Vendor"),
    col("Average Fare").alias("Average_Fare")
)

aliased_df.show()

+------+------------+
|Vendor|Average_Fare|
+------+------------+
|     1|       40.12|
|     2|       38.27|
+------+------------+



### **Join**

In [27]:
vendor_lookup = spark.createDataFrame(
    [
        (1, "Creative Mobile"),
        (2, "Curb Mobility")
    ],
    ["Vendor", "Vendor_Name"]
)

joined_df = aliased_df.join(
    vendor_lookup,
    on="Vendor",
    how="left"
)

joined_df.show()

+------+------------+---------------+
|Vendor|Average_Fare|    Vendor_Name|
+------+------------+---------------+
|     1|       40.12|Creative Mobile|
|     2|       38.27|  Curb Mobility|
+------+------------+---------------+



### **Repartition**

In [28]:
repartitioned_df = joined_df.repartition(4)

print(
    "Number of Partitions:",
    repartitioned_df.rdd.getNumPartitions()
)

Number of Partitions: 4


## **Part 6 – Spark SQL**

### **Create Temporary View**

In [29]:
clean_df.createOrReplaceTempView("taxi")

### **Top 10 Longest Trips**

In [30]:
# Query 1 - Top 10 Longest Trips
query1 = spark.sql("""
SELECT
    VendorID,
    trip_distance,
    fare_amount,
    total_amount,
    PULocationID,
    DOLocationID
FROM taxi
ORDER BY trip_distance DESC
LIMIT 10
""")

# Query 2 - Top Pickup Locations
query2 = spark.sql("""
SELECT
    PULocationID,
    COUNT(*) AS Total_Trips
FROM taxi
GROUP BY PULocationID
ORDER BY Total_Trips DESC
LIMIT 10
""")

# Query 3 - Average Fare by Payment Type
query3 = spark.sql("""
SELECT
    payment_type,
    ROUND(AVG(fare_amount),2) AS Average_Fare
FROM taxi
GROUP BY payment_type
ORDER BY payment_type
""")

# Query 4 - Peak Pickup Hour
query4 = spark.sql("""
SELECT
    HOUR(tpep_pickup_datetime) AS Pickup_Hour,
    COUNT(*) AS Total_Trips
FROM taxi
GROUP BY Pickup_Hour
ORDER BY Total_Trips DESC
LIMIT 10
""")

# Query 5 - Trips Over 20 Miles
query5 = spark.sql("""
SELECT
    VendorID,
    trip_distance,
    fare_amount,
    total_amount
FROM taxi
WHERE trip_distance > 20
ORDER BY trip_distance DESC
""")

# Query 6 - Monthly Revenue
query6 = spark.sql("""
SELECT
    MONTH(tpep_pickup_datetime) AS Month,
    ROUND(SUM(total_amount),2) AS Revenue
FROM taxi
GROUP BY Month
ORDER BY Month
""")

# Query 7 - Average Tip by Vendor
query7 = spark.sql("""
SELECT
    VendorID,
    ROUND(AVG(tip_amount),2) AS Average_Tip
FROM taxi
GROUP BY VendorID
ORDER BY VendorID
""")

# Query 8 - Average Trip Distance by Vendor
query8 = spark.sql("""
SELECT
    VendorID,
    ROUND(AVG(trip_distance),2) AS Average_Distance
FROM taxi
GROUP BY VendorID
ORDER BY VendorID
""")

# Query 9 - Passenger Count Distribution
query9 = spark.sql("""
SELECT
    passenger_count,
    COUNT(*) AS Total_Trips
FROM taxi
GROUP BY passenger_count
ORDER BY passenger_count
""")

# Query 10 - Highest Total Fare Trips
query10 = spark.sql("""
SELECT
    VendorID,
    trip_distance,
    fare_amount,
    total_amount
FROM taxi
ORDER BY total_amount DESC
LIMIT 10
""")

In [31]:
results = [

    ("Query 1 - Top 10 Longest Trips", query1),

    ("Query 2 - Top Pickup Locations", query2),

    ("Query 3 - Average Fare by Payment Type", query3),

    ("Query 4 - Peak Pickup Hour", query4),

    ("Query 5 - Trips Over 20 Miles", query5),

    ("Query 6 - Monthly Revenue", query6),

    ("Query 7 - Average Tip by Vendor", query7),

    ("Query 8 - Average Trip Distance by Vendor", query8),

    ("Query 9 - Passenger Count Distribution", query9),

    ("Query 10 - Highest Total Fare Trips", query10)

]

for title, dataframe in results:

    print(title)

    dataframe.show(truncate=False)

Query 1 - Top 10 Longest Trips


+--------+-------------+-----------+------------+------------+------------+
|VendorID|trip_distance|fare_amount|total_amount|PULocationID|DOLocationID|
+--------+-------------+-----------+------------+------------+------------+
|2       |15400.32     |28.9       |39.48       |163         |24          |
|2       |10879.28     |70.0       |98.88       |132         |224         |
|2       |1715.22      |70.0       |98.88       |132         |162         |
|1       |971.8        |21.5       |23.0        |237         |265         |
|1       |964.6        |39.5       |41.0        |71          |265         |
|2       |277.4        |33.8       |39.55       |132         |38          |
|2       |246.22       |8.6        |18.12       |163         |229         |
|2       |233.25       |1616.5     |1617.5      |168         |265         |
|2       |210.82       |500.0      |515.88      |168         |265         |
|1       |210.2        |650.0      |689.68      |132         |265         |
+--------+--

+------------+-----------+
|PULocationID|Total_Trips|
+------------+-----------+
|132         |138130     |
|237         |137093     |
|161         |136500     |
|236         |129604     |
|162         |102308     |
|186         |100737     |
|230         |100008     |
|142         |98906      |
|138         |87253      |
|239         |82391      |
+------------+-----------+

Query 3 - Average Fare by Payment Type


+------------+------------+
|payment_type|Average_Fare|
+------------+------------+
|1           |18.38       |
|2           |18.61       |
|3           |17.04       |
|4           |19.67       |
+------------+------------+

Query 4 - Peak Pickup Hour


+-----------+-----------+
|Pickup_Hour|Total_Trips|
+-----------+-----------+
|18         |198037     |
|17         |193394     |
|16         |180381     |
|15         |179080     |
|14         |173448     |
|19         |172421     |
|13         |161159     |
|12         |155458     |
|20         |150699     |
|21         |149795     |
+-----------+-----------+

Query 5 - Trips Over 20 Miles


...

+-----+-------------+
|Month|Revenue      |
+-----+-------------+
|1    |7.542614842E7|
|2    |90.47        |
|12   |235.12       |
+-----+-------------+

Query 7 - Average Tip by Vendor


+--------+-----------+
|VendorID|Average_Tip|
+--------+-----------+
|1       |3.15       |
|2       |3.57       |
+--------+-----------+

Query 8 - Average Trip Distance by Vendor


+--------+----------------+
|VendorID|Average_Distance|
+--------+----------------+
|1       |3.13            |
|2       |3.35            |
+--------+----------------+

Query 9 - Passenger Count Distribution


+---------------+-----------+
|passenger_count|Total_Trips|
+---------------+-----------+
|0              |30680      |
|1              |2134966    |
|2              |395147     |
|3              |88975      |
|4              |49598      |
|5              |33308      |
|6              |22178      |
|7              |5          |
|8              |37         |
|9              |1          |
+---------------+-----------+

Query 10 - Highest Total Fare Trips


+--------+-------------+-----------+------------+
|VendorID|trip_distance|fare_amount|total_amount|
+--------+-------------+-----------+------------+
|2       |31.95        |2221.3     |2225.3      |
|2       |233.25       |1616.5     |1617.5      |
|2       |142.62       |912.3      |940.93      |
|2       |157.25       |899.0      |900.0       |
|2       |0.21         |820.0      |821.0       |
|2       |109.75       |761.1      |775.48      |
|2       |119.46       |739.4      |771.41      |
|2       |122.47       |749.2      |758.89      |
|2       |120.76       |744.3      |753.74      |
|2       |0.11         |700.0      |715.75      |
+--------+-------------+-----------+------------+



In [32]:
import os

# Create output folder if it does not exist
os.makedirs("output", exist_ok=True)

# Save all SQL query results
query1.coalesce(1).write.mode("overwrite").option("header", True).csv("output/query1.csv")
query2.coalesce(1).write.mode("overwrite").option("header", True).csv("output/query2.csv")
query3.coalesce(1).write.mode("overwrite").option("header", True).csv("output/query3.csv")
query4.coalesce(1).write.mode("overwrite").option("header", True).csv("output/query4.csv")
query5.coalesce(1).write.mode("overwrite").option("header", True).csv("output/query5.csv")
query6.coalesce(1).write.mode("overwrite").option("header", True).csv("output/query6.csv")
query7.coalesce(1).write.mode("overwrite").option("header", True).csv("output/query7.csv")
query8.coalesce(1).write.mode("overwrite").option("header", True).csv("output/query8.csv")
query9.coalesce(1).write.mode("overwrite").option("header", True).csv("output/query9.csv")
query10.coalesce(1).write.mode("overwrite").option("header", True).csv("output/query10.csv")

print("All SQL query outputs saved successfully.")

All SQL query outputs saved successfully.


## **Part 7 – Window Functions**

In [33]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, col

### **Create Window Specification**

In [34]:
windowSpec = Window.orderBy(col("trip_distance").desc())

In [35]:
window_df = (
    clean_df
    .select(
        "VendorID",
        "trip_distance",
        "fare_amount"
    )
    .withColumn(
        "Row_Number",
        row_number().over(windowSpec)
    )
    .withColumn(
        "Rank",
        rank().over(windowSpec)
    )
    .withColumn(
        "Dense_Rank",
        dense_rank().over(windowSpec)
    )
)

In [36]:
window_df.show(20, truncate=False)

26/08/02 21:18:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/02 21:18:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/02 21:18:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/02 21:18:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/02 21:18:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/02 21:18:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/02 2

+--------+-------------+-----------+----------+----+----------+
|VendorID|trip_distance|fare_amount|Row_Number|Rank|Dense_Rank|
+--------+-------------+-----------+----------+----+----------+
|2       |15400.32     |28.9       |1         |1   |1         |
|2       |10879.28     |70.0       |2         |2   |2         |
|2       |1715.22      |70.0       |3         |3   |3         |
|1       |971.8        |21.5       |4         |4   |4         |
|1       |964.6        |39.5       |5         |5   |5         |
|2       |277.4        |33.8       |6         |6   |6         |
|2       |246.22       |8.6        |7         |7   |7         |
|2       |233.25       |1616.5     |8         |8   |8         |
|2       |210.82       |500.0      |9         |9   |9         |
|1       |210.2        |650.0      |10        |10  |10        |
|2       |207.68       |13.5       |11        |11  |11        |
|2       |176.43       |35.9       |12        |12  |12        |
|2       |157.25       |899.0      |13  

## **Part 8 – Performance Optimization**

In [37]:
import time

In [38]:
cached_df = clean_df.cache()

In [39]:
start = time.time()

clean_df.groupBy("VendorID").count().collect()

end = time.time()

before_cache = end - start

print(f"Execution Time Before Cache: {before_cache:.4f} seconds")

Execution Time Before Cache: 31.8930 seconds


In [40]:
cached_df.count()

2754895

In [41]:
start=time.time()

cached_df.groupBy("VendorID").count().show()

end=time.time()

after_cache = end - start

print(f"Execution Time After Cache: {after_cache:.4f} seconds")

+--------+-------+
|VendorID|  count|
+--------+-------+
|       1| 669555|
|       2|2085340|
+--------+-------+

Execution Time After Cache: 3.2720 seconds


In [42]:
repartition_df = cached_df.repartition(4)

In [43]:
print(
    "Number of Partitions:",
    repartition_df.rdd.getNumPartitions()
)

Number of Partitions: 4


In [44]:
repartition_df.explain(True)

== Parsed Logical Plan ==
Repartition 4, true
+- Filter atleastnnonnulls(19, VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18)
   +- Filter (fare_amount#10 >= cast(0 as double))
      +- Filter (trip_distance#4 > cast(0 as double))
         +- Deduplicate [DOLocationID#8, improvement_surcharge#15, tpep_dropoff_datetime#2, PULocationID#7, tolls_amount#14, tip_amount#13, passenger_count#3L, store_and_fwd_flag#6, extra#11, congestion_surcharge#17, total_amount#16, tpep_pickup_datetime#1, mta_tax#12, trip_distance#4, Airport_fee#18, RatecodeID#5L, VendorID#0, payment_type#9L, fare_amount#10]
            +- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4

In [45]:
print("Performance Summary")

print(f"Execution Before Cache : {before_cache:.4f} seconds")
print(f"Execution After Cache  : {after_cache:.4f} seconds")

if after_cache < before_cache:
    print("\nCaching Improved Performance.")
else:
    print("\nCaching Difference was Minimal.")

Performance Summary
Execution Before Cache : 31.8930 seconds
Execution After Cache  : 3.2720 seconds

Caching Improved Performance.


In [46]:
cached_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:47:21|  2024-01-01 00:51:58|              2|         1.13|         1|                 N|         238|         166|           1|        7.9|  1.0|    0.5|      2.5

In [47]:
print("Is Cached:", cached_df.is_cached)

Is Cached: True
